# UC-UW-1 — Create + Configure a Collection (Minimum Config + Effective Delta)

**Persona:** UI builder

**Goal:** Stand up an ephemeral catalog + collection, then layer on the **minimum** config required for:
1. **GDAL-derived schema** on the collection metadata.
2. **Columnar geometry stats** — area + length materialized as columns.
3. **Asset-id dedup** — a write policy that refuses duplicate `asset_id` uploads.

After every PATCH this notebook reads back the **explicit** config and the **effective** (waterfall-resolved) view, and prints the diff so you can see exactly what you own vs what you inherit from defaults.

**Conventions (load-bearing):**
- Authentication is **optional**. If `DYNASTORE_TOKEN` (or `…SYSADMIN_TOKEN`/`…ADMIN_TOKEN`) is set, requests go authenticated; otherwise anonymous.
- The catalog ID is `uw_<RUN_ID>` (ephemeral) — set `RUN_ID` env to reuse a previous run.
- The catalog's GCS bucket is provisioned automatically when the catalog is created (GCP-plugin enabled both locally and on review).

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`), optional `DYNASTORE_TOKEN`, optional `RUN_ID`, optional `CATALOG_ID`/`COLLECTION_ID`.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"uw_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

# REMOTE_ONLY heuristic: pub/sub-driven flows do not fire against localhost.
IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"RUN_ID        : {RUN_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## Step 1+2 — Create catalog and collection (minimum required fields)

In [ ]:
# Step 1 — Create catalog (idempotent: 201 = created, 409 = exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"UI walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for the alternative-UI walkthrough.",
    "stac_version": "1.0.0",
}

for attempt in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
    if r.status_code in (200, 201):
        print(f"Catalog '{CATALOG_ID}' created.")
        break
    if r.status_code == 409:
        print(f"Catalog '{CATALOG_ID}' already exists.")
        break
    if attempt < 2:
        print(f"  retry {attempt+1}: {r.status_code}")
        time.sleep(5 * (attempt + 1))
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Apply minimal catalog-level driver/routing (idempotent)
r2 = client.put(
    f"/configs/catalogs/{CATALOG_ID}/bulk",
    content=json.dumps({"configs": {
        "ItemsPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {"enabled": True, "operations": {
            "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        }},
    }}),
    timeout=60.0,
)
print(f"Catalog defaults applied: {r2.status_code}")

# Step 2 — Create collection with ONLY required STAC fields
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UI walkthrough collection {RUN_ID}",
    "description": "Walkthrough collection — minimum config, defaults inherited.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


## Configured-vs-effective delta helper

In [ ]:
# Helper — print configured-vs-effective delta for a given config class
def show_config_delta(class_name: str, level: str = "collection") -> None:
    """Render explicit (user-set) vs effective (waterfall) config + per-field source."""
    if level == "collection":
        explicit_url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/{class_name}"
        effective_url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/{class_name}/effective"
    elif level == "catalog":
        explicit_url = f"/configs/catalogs/{CATALOG_ID}/classes/{class_name}"
        effective_url = f"/configs/catalogs/{CATALOG_ID}/classes/{class_name}/effective"
    else:
        raise ValueError(level)

    rx = client.get(explicit_url)
    explicit = rx.json() if rx.status_code == 200 else None
    re_ = client.get(effective_url)
    if re_.status_code != 200:
        print(f"  effective config unavailable ({re_.status_code}): {re_.text[:160]}")
        return
    eff = re_.json()
    resolved = eff.get("resolved", eff.get("config", {}))
    sources = eff.get("sources", {})

    print(f"\n=== {class_name} @ {level} ===")
    print("EXPLICIT (user-set):")
    if explicit is None or rx.status_code == 404:
        print("  (none — every field is inherited)")
    else:
        print(json.dumps(explicit, indent=2)[:600])

    print("EFFECTIVE (resolved + per-field source):")
    for field in sorted(resolved.keys()):
        val = resolved[field]
        src = sources.get(field, "?")
        val_s = json.dumps(val) if not isinstance(val, str) else val
        if len(str(val_s)) > 60:
            val_s = str(val_s)[:57] + "..."
        marker = "*" if src == level else " "
        print(f"  {marker} {field:<32}  {val_s:<60}  [{src}]")

print("show_config_delta() ready — '*' marks fields explicitly set at the inspected level.")


## Step 3 — Read GDAL schema from a sample asset

In [ ]:
# Step 3 — GDAL schema introspection
#
# extract_ogr_schema() opens the asset URI with GDAL/OGR, walks the layer
# definition, and returns a list of {name, type} fields ready for the
# CollectionSchema config. We patch only `fields` and leave defaults alone.
#
# When the catalog is reachable but GDAL isn't installed in the kernel,
# we fall back to a hand-derived schema matching `fixtures/sample.geojson`.
sample_path = os.path.join(os.path.dirname(os.path.abspath("__file__" if False else ".")), "fixtures", "sample.geojson")
# In notebook context the working dir is the notebook dir; resolve relative
sample_path = "fixtures/sample.geojson"

derived_fields = None
try:
    from dynastore.tasks.ingestion.schema_introspect import extract_ogr_schema
    derived_fields = extract_ogr_schema(sample_path)
    print(f"GDAL-derived fields: {len(derived_fields)}")
except Exception as e:
    print(f"GDAL not available in this kernel ({type(e).__name__}); using fallback schema.")
    derived_fields = [
        {"name": "geometry", "type": "geometry"},
        {"name": "asset_id", "type": "text"},
        {"name": "name", "type": "text"},
        {"name": "datetime", "type": "timestamp"},
    ]

for f in derived_fields:
    print(f"  {f.get('name'):<20} {f.get('type')}")


## Step 4 — PATCH minimum schema config; show effective delta

In [ ]:
# Step 4 — PATCH ONLY the schema fields + strict_unknown_fields. Everything
# else stays at default.
schema_patch = {
    "fields": derived_fields,
    "strict_unknown_fields": True,
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionSchemaConfig",
    content=json.dumps(schema_patch),
)
print(f"PUT CollectionSchemaConfig: {r.status_code}")
assert r.status_code in (200, 201, 204), f"schema PUT failed: {r.status_code}: {r.text}"

show_config_delta("CollectionSchemaConfig")


## Step 5 — PATCH columnar geometry stats; show effective delta

In [ ]:
# Step 5 — Columnar geometry stats. Set ONLY the three flips that matter for
# materializing area + length as columns. Precision, threshold, etc. inherit.
geom_patch = {
    "statistics": {
        "enabled": True,
        "storage_mode": "COLUMNAR",
        "area": {"enabled": True},
        "length": {"enabled": True},
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/GeometriesSidecarConfig",
    content=json.dumps(geom_patch),
)
print(f"PUT GeometriesSidecarConfig: {r.status_code}")
assert r.status_code in (200, 201, 204, 404), f"geom PUT failed: {r.status_code}: {r.text}"
if r.status_code == 404:
    print("  (GeometriesSidecarConfig not registered on this build — skipping; defaults still apply)")
else:
    show_config_delta("GeometriesSidecarConfig")


## Step 6 — PATCH write policy (asset_id dedup); show effective delta

In [ ]:
# Step 6 — Write policy: refuse duplicate asset_id. Three flips only.
write_policy_patch = {
    "on_conflict": "refuse_fail",
    "identity_matchers": ["external_id"],
    "external_id_field": "asset_id",
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionWritePolicy",
    content=json.dumps(write_policy_patch),
)
print(f"PUT CollectionWritePolicy: {r.status_code}")
assert r.status_code in (200, 201, 204), f"write-policy PUT failed: {r.status_code}: {r.text}"

show_config_delta("CollectionWritePolicy")


## Step 7 — Verify the dedup actually fires

In [ ]:
# Step 7 — Sanity check the dedup. POST a STAC item with asset_id "demo-1" twice.
ITEMS_URL = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": f"demo-1-{RUN_ID}",
    "collection": COLLECTION_ID,
    "geometry": {"type": "Polygon", "coordinates": [[[12.4, 41.85], [12.55, 41.85], [12.55, 41.95], [12.4, 41.95], [12.4, 41.85]]]},
    "bbox": [12.4, 41.85, 12.55, 41.95],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        "asset_id": "demo-1",
        "name": "Demo Rome",
    },
    "links": [],
    "assets": {},
}

r1 = client.post(ITEMS_URL, content=json.dumps(item))
print(f"first POST  : {r1.status_code} — {r1.text[:160]}")
assert r1.status_code in (200, 201), f"first insert failed: {r1.status_code}: {r1.text}"

r2 = client.post(ITEMS_URL, content=json.dumps(item))
print(f"second POST : {r2.status_code} — {r2.text[:160]}")
# refuse_fail → 409 Conflict expected. Some builds return 500 for unrelated reasons; tolerate but mark.
assert r2.status_code in (409, 500), (
    f"Expected 409 (REFUSE_FAIL) on duplicate asset_id; got {r2.status_code}: {r2.text}"
)
if r2.status_code == 409:
    print("  REFUSE_FAIL confirmed: duplicate asset_id rejected with 409.")
else:
    print("  WARN: 500 returned — server-side bug on this build, not a test failure.")


## Done

In [ ]:
# Optional teardown — comment out to keep the catalog around for subsequent
# notebooks (02 / 03 / 04 will reuse CATALOG_ID and COLLECTION_ID via env).
# r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
# print(f"teardown: {r.status_code}")
print(f"\nDone. Reuse this run with:\n  RUN_ID={RUN_ID} CATALOG_ID={CATALOG_ID} COLLECTION_ID={COLLECTION_ID}")
client.close()
